In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sqlite3                                              # 读 SQLite 数据库（Python 自带，不用装）
import json                                                 # 写 metrics.json、特征清单
import joblib                                               # 保存整条 Pipeline

from sklearn.model_selection import train_test_split        # 划分数据集用的
from sklearn.pipeline import Pipeline                       # 搭建流水线
from sklearn.compose import ColumnTransformer               # 数值列、类别列分开处理
from sklearn.impute import SimpleImputer                    # 填补缺失值
from sklearn.preprocessing import OneHotEncoder             # 对数据做独热编码
from sklearn.preprocessing import StandardScaler            # 对数据做标准化
from sklearn.linear_model import LogisticRegression         # 逻辑回归
from sklearn.ensemble import RandomForestClassifier         # 随机森林
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score          # 三个常用指标
from sklearn.metrics import confusion_matrix, classification_report          # 混淆矩阵、分类报告

pd.set_option("display.max_columns", 40)                    # 列多，全部显示
pd.set_option("display.width", 120)

In [2]:
SEED = 42 
DATA = "../data/"
DB = DATA + "incident.db"
# 设置后期使用的标签列的列名
TARGET = "label"

In [5]:
# with 语句: 自动打开数据库连接，代码块结束后自动关闭连接，不用手动写 conn.close()，防止忘记关闭
with sqlite3.connect(DB) as conn:
    # 从 sqlite 的系统表 sqlite_master 查询，拿到数据库里所有普通数据表的名字
    tables = pd.read_sql("select name from sqlite_master where type='table'", conn)
    print(tables)

    # 循环遍历每一张表的表名，查看表里面的信息
    for name in tables['name']:
        # PRAGMA table_info(表名)：SQLite 专属命令，查询这张表的字段信息（列名、类型、是否主键等）
        cols = pd.read_sql(f"PRAGMA table_info({name})",conn)
        # COUNT(*) 统计表总行数；取出结果里n这一列第0个值，就是行数
        n = pd.read_sql(f"select count(1) as n from {name}",conn)["n"][0]
        # 格式化打印输出：表名左对齐占12字符、行数右对齐、列数量，最后拼接所有列名
        print(f"{name:<12}{n:>8}行 {len(cols):>2}列: ", ", ".join(cols["name"]))

          name
0    residents
1       visits
2  lab_results
residents      36000行  7列:  resident_id, gender, birth_date, education, income_level, district, enroll_date
visits        101297行 23列:  visit_id, resident_id, visit_date, height_cm, weight_kg, waist_cm, sbp, dbp, smoking_status, drinking_freq, exercise_min_week, sleep_hours, veg_fruit_daily, family_history_dm, hypertension_dx, hyperlipidemia_dx, gdm_history, polyuria, polydipsia, weight_loss_unexplained, known_diabetes, created_at, chronic_mgmt_tag
lab_results    79358行  6列:  lab_id, resident_id, sample_date, report_date, fpg_mmol_l, hba1c_pct


In [9]:
# parse_dates=[...] 在读的时候就把日期列转成 datetime，后面做日期加减不用再 pd.to_datetime
with sqlite3.connect(DB) as conn:
    residents = pd.read_sql("select * from residents",conn,parse_dates=['birth_date','enroll_date'])
    visits = pd.read_sql("select * from visits",conn,parse_dates=['visit_date',])
    labs = pd.read_sql("select * from lab_results",conn)
    print(visits)
    print(labs)
    print(residents.shape,visits.shape,labs.shape)

         visit_id resident_id visit_date  height_cm  weight_kg  waist_cm    sbp   dbp smoking_status drinking_freq  \
0       V00000001    R0000131 2023-01-01      164.1       61.4      78.6  100.0  65.0          never      frequent   
1       V00000002    R0000342 2023-01-01      162.2       73.3      91.9  115.0  62.0          never    occasional   
2       V00000003    R0000387 2023-01-01      161.5       56.0       NaN  118.0  72.0        current         never   
3       V00000004    R0000736 2023-01-01      172.0       70.2      80.6  132.0  84.0        current    occasional   
4       V00000005    R0001057 2023-01-01      156.0       48.5      63.1  125.0  69.0          never         never   
...           ...         ...        ...        ...        ...       ...    ...   ...            ...           ...   
101292  V00101293    R0034982 2025-06-30      161.6       47.9      76.3   75.0  58.0          never    occasional   
101293  V00101294    R0035219 2025-06-30      161.5     

In [12]:
visits.head(3).T

,0,1,2
visit_id,V00000001,V00000002,V00000003
resident_id,R0000131,R0000342,R0000387
visit_date,2023-01-01 00:00:00,2023-01-01 00:00:00,2023-01-01 00:00:00
height_cm,164.1,162.2,161.5
weight_kg,61.4,73.3,56.0
waist_cm,78.6,91.9,NaN
sbp,100.0,115.0,118.0
dbp,65.0,62.0,72.0
smoking_status,never,never,current
drinking_freq,frequent,occasional,never


In [16]:
visits.describe(include="number").T[['count', 'mean', 'min', '50%', 'max']].round(1)

,count,mean,min,50%,max
height_cm,101297.0,162.3,1.4,163.0,191.3
weight_kg,101297.0,64.7,31.1,64.0,198.4
waist_cm,86036.0,81.7,39.1,81.6,123.4
sbp,101297.0,120.5,0.0,121.0,176.0
dbp,101297.0,74.5,36.0,74.0,111.0
exercise_min_week,101297.0,132.1,0.0,82.0,10080.0
sleep_hours,93165.0,6.9,3.0,6.9,11.3
veg_fruit_daily,101297.0,0.6,0.0,1.0,1.0
hypertension_dx,101297.0,0.1,0.0,0.0,1.0
hyperlipidemia_dx,101297.0,0.1,0.0,0.0,1.0


In [ ]:
# 查看 visits 表中每一列的确实比例
print(visits.isna().mean().round(3).sort_values(ascending=False).head(4))

gdm_history                0.485
waist_cm                   0.151
sleep_hours                0.080
weight_loss_unexplained    0.030
dtype: float64


In [18]:
# `duplicated()` 判断是不是和前面一模一样的整行
print("完全重复的行：",visits.duplicated().sum())

完全重复的行： 0


In [19]:
# 根据性别统计行数
print(residents['gender'].value_counts().to_dict())

# 吸烟状态
print(visits['smoking_status'].value_counts().to_dict())

# 饮酒频率统计
print(visits["drinking_freq"].value_counts().to_dict())

# 删除完全重复的
visits = visits.drop_duplicates()
# 统一性别的类别标签，最终留下：F / M (注：底部被截断)

{'F': 18276, 'M': 17032, 'male': 186, 'female': 174, '女': 172, '男': 160}
{'never': 66495, 'current': 22022, 'former': 8615, 'never ': 961, '从不': 956, 'Never': 930, 'Current': 473, '吸烟': 471, 'Former': 203, '已戒': 171}
{'never': 49524, 'occasional': 34237, 'frequent': 14436, '不喝': 1534, '偶尔': 1097, '经常': 469}


In [28]:
# 删除完全重复的
visits = visits.drop_duplicates()

# 统一性别的类别标签，最终留下：F / M
residents['gender'] = residents['gender'].replace(
    {"male":"M","男":"M","female":"F","女":"F"}
)

# 统一吸烟状态的类别标签，最终留下：英文 never，current，former
visits['smoking_status'] = visits['smoking_status'].replace(
    {'从不':"never", 
    '吸烟':"current", 
    '已戒':'former', 
    'Never':'never', 
    'Current':'current', 
    'Former':'former', 
    'never ':'never'}
    )

# 统一饮酒频率的类别标签，最终留下：never\occasional\frequent
# 【注意】原代码末尾缺少一个右括号 `)`
visits["drinking_freq"] = visits["drinking_freq"].replace(
    {"不喝":"never", 
    "偶尔":"occasional", 
    "经常":"frequent"}
    )

print(residents['gender'].value_counts().to_dict())
print(visits['smoking_status'].value_counts().to_dict())
print(visits["drinking_freq"].value_counts().to_dict())

{'F': 18622, 'M': 17378}
{'never': 69342, 'current': 22966, 'former': 8989}
{'never': 51058, 'occasional': 35334, 'frequent': 14905}


In [29]:
# 按米录入的身高 → 厘米
visits.loc[visits['height_cm']<3,'height_cm'] *=100
visits[visits['height_cm']<3]

# 按斤录入的体重 → 公斤
visits.loc[visits['weight_kg'] > 150,'weight_kg'] /= 2

# sbp = 收缩压（高压），血压不可能等于 0
visits.loc[visits['sbp'] == 0,'sbp'] = np.nan

visits.loc[visits['exercise_min_week'] > 3000,'exercise_min_week'] = np.nan

visits[
    [
    'height_cm', 
    'weight_kg', 
    'sbp', 
    'exercise_min_week'
    ]
    ].describe().T[
        ['min', 
        '50%', 
        'max']
        ]

,min,50%,max
height_cm,134.4,163.0,191.3
weight_kg,31.1,64.0,149.7
sbp,62.0,121.0,176.0
exercise_min_week,0.0,82.0,2162.0
